[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![在 LangChain Academy 中打开](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)


## LangChain 学院

欢迎来到 LangChain 学院！

-----

### **背景介绍**

在 LangChain，我们的目标是让构建大型语言模型（LLM）应用变得简单。您可以构建的一种 LLM 应用就是**智能代理（Agent）**。构建智能代理非常令人兴奋，因为它们能够自动化以前不可能完成的各种任务。

然而，在实践中，构建能够可靠执行这些任务的系统极其困难。在我们与用户合作将智能代理投入生产的过程中，我们发现通常需要更多的**控制**。例如，您可能需要智能代理始终优先调用某个特定的工具，或者根据其状态使用不同的提示词。

为了解决这个问题，我们构建了 [**LangGraph**](https://langchain-ai.github.io/langgraph/) —— 一个用于构建智能代理和多智能体应用的框架。LangGraph 独立于 LangChain 包，其核心设计理念是帮助开发者为智能代理工作流添加更好的**精确性**和**控制力**，使其适合现实世界系统的复杂性。

-----

### **课程结构**

本课程由一系列模块组成，每个模块都专注于一个与 LangGraph 相关的主题。您会看到每个模块都有一个文件夹，其中包含一系列**笔记本（notebooks）**。每本笔记本都配有视频，以帮助您理解概念，但这些笔记本也是独立的，这意味着它们包含了详细的解释，可以脱离视频独立观看。每个模块文件夹还包含一个 `studio` 文件夹，其中包含一组图（graphs），这些图可以加载到 [**LangGraph Studio**](https://github.com/langchain-ai/langgraph-studio) 中，这是我们用于构建 LangGraph 应用的集成开发环境（IDE）。

-----

### **设置**

在开始之前，请按照 `README` 文件中的说明创建环境并安装依赖项。

-----

### **聊天模型**

在本课程中，我们将使用**聊天模型（Chat Models）**，它们的功能是接收一系列消息作为输入，并以聊天消息作为输出。LangChain 本身不托管任何聊天模型，而是依赖于第三方集成。 [suspicious link removed] 是 LangChain 中支持的第三方聊天模型集成列表！默认情况下，课程将使用 [suspicious link removed]，因为它既流行又性能出色。正如前面提到的，请确保您已设置好 `OPENAI_API_KEY`。

我们将检查您的 `OPENAI_API_KEY` 是否已设置；如果未设置，系统会提示您输入。

In [ ]:
# 安装必要的依赖包
# %%capture --no-stderr 用于隐藏安装过程中的输出信息
# %pip install 是 Jupyter 中安装 Python 包的命令
# --quiet 参数减少输出信息
# -U 参数表示升级到最新版本
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langchain_community tavily-python

In [ ]:
# 导入必要的模块
import os, getpass

def _set_env(var: str):
    """
    设置环境变量的辅助函数
    
    参数:
        var (str): 要设置的环境变量名称
    
    功能:
        - 检查环境变量是否已存在
        - 如果不存在，则提示用户输入并设置
    """
    if not os.environ.get(var):  # 检查环境变量是否已设置
        os.environ[var] = getpass.getpass(f"{var}: ")  # 安全地获取用户输入

# 设置 OpenAI API 密钥
# 这是使用 OpenAI 模型所必需的
_set_env("OPENAI_API_KEY")

[这里](https://python.langchain.com/v0.2/docs/how_to/#chat-models)是一个有用的指南，介绍了您可以使用聊天模型做的所有事情，但我们在下面会展示一些重点。如果您按照 README 中的说明运行了 `pip install -r requirements.txt`，那么您已经安装了 `langchain-openai` 包。有了这个包，我们可以实例化我们的 `ChatOpenAI` 模型对象。如果您是第一次注册 API，您应该会收到[免费积分](https://community.openai.com/t/understanding-api-limits-and-free-tier/498517)，可以应用于任何模型。您可以[在这里](https://openai.com/api/pricing/)查看各种模型的定价。笔记本将默认使用 `gpt-4o`，因为它在质量、价格和速度之间取得了良好的平衡[更多信息请参见这里](https://help.openai.com/en/articles/7102672-how-can-i-access-gpt-4-gpt-4-turbo-gpt-4o-and-gpt-4o-mini)，但您也可以选择价格较低的 `gpt-3.5` 系列模型。

聊天模型有几个[标准参数](https://python.langchain.com/v0.2/docs/concepts/#chat-models)可以设置。最常见的两个是：

* `model`：模型名称
* `temperature`：采样温度

`Temperature` 控制模型输出的随机性或创造性，其中低温度（接近 0）产生更确定性和专注的输出。这适合需要准确性或事实性响应的任务。高温度（接近 1）适合创造性任务或生成多样化响应。 

In [ ]:
# 导入 ChatOpenAI 类
from langchain_openai import ChatOpenAI

# 创建 GPT-4o 聊天模型实例
# model="gpt-4o": 使用 GPT-4o 模型，这是 OpenAI 的最新模型
# temperature=0: 设置温度为 0，使输出更加确定性和一致
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)

# 创建 GPT-3.5 Turbo 聊天模型实例
# model="gpt-3.5-turbo-0125": 使用 GPT-3.5 Turbo 模型（2025年1月版本）
# temperature=0: 同样设置为 0，确保输出的一致性
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)

LangChain 中的聊天模型有许多[默认方法](https://python.langchain.com/v0.2/docs/concepts/#runnable-interface)。在大多数情况下，我们将使用：

* `stream`：流式返回响应的块
* `invoke`：在输入上调用链

如前所述，聊天模型接受[消息](https://python.langchain.com/v0.2/docs/concepts/#messages)作为输入。消息具有一个角色（描述谁在说消息）和一个内容属性。我们稍后会更多地讨论这个问题，但这里让我们先展示基础知识。

In [ ]:
# 导入 HumanMessage 类，用于创建人类用户的消息
from langchain_core.messages import HumanMessage

# 创建一个人类消息
# content: 消息内容
# name: 发送者的名称（可选）
msg = HumanMessage(content="Hello world", name="Lance")

# 创建消息列表
# 聊天模型通常接受消息列表作为输入
messages = [msg]

# 使用消息列表调用模型
# invoke() 方法会发送消息给模型并返回响应
gpt4o_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 11, 'total_tokens': 20}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_157b3831f5', 'finish_reason': 'stop', 'logprobs': None}, id='run-d3c4bc85-ef14-49f6-ba7e-91bf455cffee-0', usage_metadata={'input_tokens': 11, 'output_tokens': 9, 'total_tokens': 20})

我们得到一个 `AIMessage` 响应。另外，请注意我们可以直接用字符串调用聊天模型。当字符串作为输入传递时，它会被转换为 `HumanMessage`，然后传递给底层模型。


In [ ]:
# 直接使用字符串调用模型
# 当传入字符串时，LangChain 会自动将其转换为 HumanMessage
gpt4o_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18}, 'model_name': 'gpt-4o-2024-05-13', 'system_fingerprint': 'fp_157b3831f5', 'finish_reason': 'stop', 'logprobs': None}, id='run-d6f6b682-e29a-44de-b45e-79fad1e405e5-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18})

In [ ]:
# 使用 GPT-3.5 Turbo 模型调用
# 同样可以直接传入字符串
gpt35_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 9, 'total_tokens': 18}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-c75d3f0f-2d71-47be-b14c-42b8dd2b4b08-0', usage_metadata={'input_tokens': 9, 'output_tokens': 9, 'total_tokens': 18})

所有聊天模型的接口都是一致的，模型通常在每个笔记本启动时初始化一次。

因此，如果您强烈偏好另一个提供商，您可以轻松地在模型之间切换，而无需更改下游代码。


## 搜索工具

您还会在 README 中看到 [Tavily](https://tavily.com/)，这是一个为 LLM 和 RAG 优化的搜索引擎，旨在提供高效、快速和持久的搜索结果。如前所述，注册很容易，并提供慷慨的免费层级。一些课程（在模块 4 中）将默认使用 Tavily，但当然，如果您想为自己修改代码，也可以使用其他搜索工具。

In [ ]:
# 设置 Tavily API 密钥
# 这是使用 Tavily 搜索功能所必需的
_set_env("TAVILY_API_KEY")

In [ ]:
# 导入 Tavily 搜索结果工具
from langchain_community.tools.tavily_search import TavilySearchResults

# 创建 Tavily 搜索实例
# max_results=3: 设置最大返回结果数量为 3
tavily_search = TavilySearchResults(max_results=3)

# 执行搜索
# 搜索 "What is LangGraph?" 并获取结果
search_docs = tavily_search.invoke("What is LangGraph?")

In [ ]:
# 显示搜索结果
# 这将输出搜索到的文档列表，包含 URL 和内容摘要
search_docs

[{'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'content': 'LangGraph is a library within the LangChain ecosystem designed to tackle these challenges head-on. LangGraph provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured manner.'},
 {'url': 'https://langchain-ai.github.io/langgraph/',
  'content': 'Overview LangGraph is a library for building stateful, multi-actor applications with LLMs, used to create agent and multi-agent workflows. Compared to other LLM frameworks, it offers these core benefits: cycles, controllability, and persistence. LangGraph allows you to define flows that involve cycles, essential for most agentic architectures, differentiating it from DAG-based solutions. As a ...'},
 {'url': 'https://www.youtube.com/watch?v=nmDFSVRnr4Q',
  'content': 'LangGraph is an extension of LangChain enabling Multi-Agent conversation and cyclic chains. This video explains the basics of LangGraph and codesLang